# Simple KFP + MLflow smoke test
Ovaj notebook provjerava da Kubeflow Pipelines i MLflow rade zajedno iz integriranog Kubeflow notebooka.
Primjer je namjerno minimalan: priprema dataset, trenira jedan model i logira parametre, metrike i model u MLflow.

In [ ]:
%pip uninstall -y kfp kfp-pipeline-spec kfp-server-api protobuf
%pip install --quiet --force-reinstall kfp==2.10.0 protobuf==4.25.5 mlflow scikit-learn pandas boto3

## Imports and configuration
`MLFLOW_URI` mora pokazivati na in-cluster MLflow servis, a `NAMESPACE` mora biti namespace notebooka iz kojeg pokrećeš pipeline runove.

In [ ]:
from kfp import Client, dsl
from kfp.dsl import Dataset, Input, Model, Output, component
import kfp

MLFLOW_URI = "http://mlflow-service.mlflow.svc.cluster.local:5000"
NAMESPACE = "kubeflow-user-example-com"
EXPERIMENT_NAME = "kubeflow-mlflow-smoke-test"

## Data preparation component
Prva komponenta dohvaća California Housing dataset i sprema ga kao CSV artefakt za sljedeći korak u pipelineu.

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "scikit-learn"]
)
def prepare_data(output_dataset: Output[Dataset]):
    import pandas as pd
    from sklearn.datasets import fetch_california_housing

    housing = fetch_california_housing()
    dataframe = pd.DataFrame(housing.data, columns=housing.feature_names)
    dataframe["target"] = housing.target

    dataframe.to_csv(output_dataset.path, index=False)

## Training and MLflow logging component
Druga komponenta trenira jedan jednostavan `GradientBoostingRegressor`, logira parametre, metrike i model u MLflow, te zapisuje osnovne metrike u izlazni model artefakt.

In [ ]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "scikit-learn", "mlflow", "boto3"]
)
def train_and_log_model(
    input_dataset: Input[Dataset],
    mlflow_uri: str,
    experiment_name: str,
    output_model: Output[Model],
):
    import os
    import pandas as pd
    import mlflow
    from mlflow.models import infer_signature
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.model_selection import train_test_split

    os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio.mlflow.svc.cluster.local:9000"
    os.environ["AWS_ACCESS_KEY_ID"] = "bkmZYuZ9gW"
    os.environ["AWS_SECRET_ACCESS_KEY"] = "p3DCt5rRLh"
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

    mlflow.set_tracking_uri(mlflow_uri)
    mlflow.set_experiment(experiment_name)

    dataframe = pd.read_csv(input_dataset.path)
    features = dataframe.drop(columns=["target"])
    target = dataframe["target"]

    x_train, x_test, y_train, y_test = train_test_split(
        features,
        target,
        test_size=0.2,
        random_state=42,
    )

    model = GradientBoostingRegressor(
        learning_rate=0.1,
        n_estimators=100,
        random_state=42,
    )

    with mlflow.start_run() as run:
        model.fit(x_train, y_train)
        predictions = model.predict(x_test)

        metrics = {
            "r2_score": r2_score(y_test, predictions),
            "mae": mean_absolute_error(y_test, predictions),
            "mse": mean_squared_error(y_test, predictions),
        }

        mlflow.log_params({
            "model_type": "GradientBoostingRegressor",
            "learning_rate": 0.1,
            "n_estimators": 100,
            "test_size": 0.2,
        })
        mlflow.log_metrics(metrics)

        signature = infer_signature(x_test, predictions)
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=x_test.head(5),
        )

        output_model.metadata["run_id"] = run.info.run_id
        output_model.metadata["r2_score"] = float(metrics["r2_score"])
        output_model.metadata["mae"] = float(metrics["mae"])
        output_model.metadata["mse"] = float(metrics["mse"])

## Pipeline definition
Pipeline spaja pripremu dataseta i trening komponentu. Cilj je jednostavno potvrditi da KFP može pokrenuti job i da se rezultat vidi u MLflowu.

In [ ]:
@dsl.pipeline(name="kubeflow-mlflow-smoke-test")
def smoke_test_pipeline():
    data_task = prepare_data()
    train_and_log_model(
        input_dataset=data_task.outputs["output_dataset"],
        mlflow_uri=MLFLOW_URI,
        experiment_name=EXPERIMENT_NAME,
    )

## Submit a pipeline run
Ovaj korak radi submit pipeline runa direktno iz notebooka. Za to notebook i dalje treba imati ispravan PodDefault za pristup Kubeflow Pipelines API-ju.

In [ ]:
client = Client(namespace=NAMESPACE)
run = client.create_run_from_pipeline_func(
    smoke_test_pipeline,
    enable_caching=False,
    arguments={},
)

print(f"Started run: {run.run_id}")

## What to verify after the run
Nakon što run završi, provjeri u Kubeflow Pipelines UI da je workflow prošao i u MLflow UI da postoji novi run u eksperimentu `kubeflow-mlflow-smoke-test`.

In [ ]:
print("Pipeline submitted. Open Kubeflow Pipelines and MLflow to verify the run.")
print(f"Namespace: {NAMESPACE}")
print(f"MLflow tracking URI: {MLFLOW_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")